# The agent says it succeeded. Did the business?

Three LangGraph refund agents answer the same synthetic refund requests. Each looks good on at least one usual signal: cost, speed, its own report, or an LLM judge. Only Snowflake, which can join each run to orders, policies, payments and the ledger, shows which agent actually did the right thing with the money.

| Agent | Model (Cortex) | Policy | Reply | Refund tool |
| --- | --- | --- | --- | --- |
| Lean | `claude-haiku-5-5` | cached snapshot (30 days) | the model's reply | v1 |
| Baseline | `claude-sonnet-5-5` | live (14 days since Nov 7) | reply template v1 | v1 |
| Fixed | `claude-sonnet-5-5` | live | reply template v2 | v2 |

Tool v1 refunds the whole order even when the customer asked for part of it, and ignores unsettled payments. Tool v2 refunds the requested amount and refuses unsettled payments. Reply template v1 tells the customer the refund is complete whenever the agent decided to refund, whatever the tool returned; v2 sends the model's own reply.

> **The flaws are deliberate.** Lean and Baseline are handicapped on purpose, each with a planted flaw in code that a real team ships by accident: Lean reads a stale cached policy, Baseline's reply template always reports success, and both use the buggy tool v1. Fixed is the control. This is not a comparison of `claude-haiku-5-5` and `claude-sonnet-5-5`. It tests whether cost, self-reported success and LLM judges can spot these flaws, and shows that only business truth reliably does. The flaws are scripted, but every model decision and score below comes from live runs.

The simulation uses a fixed scenario set (`refund_agent.lab.SCENARIOS`): 9 orders and 10 requests chosen so each flaw gets a chance to trigger.

**Steps:** 1. Build the agents. 2. Stream events and trace with TruLens and Langfuse. 3. Join to business data. 4. Find false successes. 5. Diagnose root causes. 6. Compare. Then TruLens vs Langfuse, and cleanup.

Every order, payment and refund is synthetic. Nothing touches a real payment system.

## 1. Build the agents

One graph, four steps: `fetch_facts → decide → refund_tool → respond`. The `decide` and `respond` steps call a Cortex model through `langchain-snowflake`'s `ChatSnowflake`. Every model call records its model, tokens and latency, so cost is measured rather than assumed.

The agent code lives in `src/refund_agent/agents.py`, shared with the Streamlit app.

In [ ]:
import logging
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)
sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
import snowflake.connector
from langchain_snowflake import ChatSnowflake
from snowflake.snowpark import Session

from refund_agent import config
from refund_agent.agents import AGENTS, build_agent, run_request

# Account settings come from REFUND_LAB_* environment variables (see src/refund_agent/config.py).
conn = snowflake.connector.connect(
    connection_name=config.CONNECTION, paramstyle="qmark",
    session_parameters={"PYTHON_CONNECTOR_QUERY_RESULT_FORMAT": "json"})
session = Session.builder.config("connection", conn).create()
identity = session.sql("SELECT CURRENT_ORGANIZATION_NAME() || '-' || CURRENT_ACCOUNT()").collect()[0][0]
assert identity == config.ACCOUNT, f"connected to {identity}, expected REFUND_LAB_ACCOUNT={config.ACCOUNT}"

chat = lambda model: ChatSnowflake(model=model, session=session, temperature=0)
graphs = {agent: build_agent(agent, chat) for agent in AGENTS}
pd.DataFrame(AGENTS).T[["model", "policy_source", "tool"]]

In [ ]:
# One request, all three agents: a 20-day-old order after the window shrank to 14 days.
live_policy = {"policy_id": "P_2026_14D", "max_age_days": 14}
request = {"case_id": "TRY", "order_id": "TRY_O1", "amount_cents": 4000, "order_total_cents": 4000,
           "age_days": 20, "payment_status": "settled", "processor_status": "ok",
           "decision_at": "2026-11-08T12:00:00+00:00", "live_policy": live_policy}
tries = [run_request(request, agent, graphs[agent], cohort="TRY", trial=1) for agent in AGENTS]
pd.DataFrame(tries)[["variant", "agent_policy_id", "tool_status", "ledger_effect_cents", "claim",
                     "model_cost_credits", "agent_latency_ms", "reply"]]

## 2. Stream events and trace with both observers

A simulation creates the fixed scenario set's orders and payments, then sends each refund request to each agent.

- **Snowpipe Streaming** (high-performance architecture) writes `RUNS`, `TRACE_EVENTS` and `LLM_CALLS` through their default `<TABLE>-STREAMING` pipes. It uses the key-pair service user `REFUND_LAB_STREAMER`, set up in `sql/streaming_setup.sql`.
- **TruLens** records each agent as a version of the External Agent `REFUND_AGENT_LAB.DEMO.REFUND_AGENT_LAB`, with one live run per agent per simulation.
- **Langfuse**, self-hosted on Snowpark Container Services, gets the same runs through LangChain's `CallbackHandler`. Each trace ID is derived from the business run ID. Langfuse's own LLM-as-a-judge evaluators score a `judge_input` step on each trace.

Langfuse must be running: `CALL REFUND_AGENT_LAB.LANGFUSE.MANAGE_LANGFUSE('resume','http://localhost:3000')`.

In [ ]:
import time
import uuid

from refund_agent.lab import StreamingSink, simulate
from refund_agent.observers import LangfuseObserver, TruLensObserver

cohort = "APP_" + uuid.uuid4().hex[:16].upper()

env = dict(line.split("=", 1) for line in
           (Path.home() / ".langfuse_spcs.env").read_text().splitlines() if line)
ingress_token = conn._rest._token_request("ISSUE")["data"]["sessionToken"]
trulens = TruLensObserver(session)
# No judge_llm: Langfuse's own evaluators score each run (scripts/langfuse_native_evals.py).
langfuse = LangfuseObserver(config.LANGFUSE_HOST, env["LF_INIT_PUBLIC_KEY"], env["LF_INIT_SECRET_KEY"],
                            ingress_token)
sink = StreamingSink(cohort, str(Path.home() / ".snowflake/keys/refund_lab_streamer.p8"))
# With TruLens on, each model call is also a TruLens cost span.
graphs = {agent: build_agent(agent, TruLensObserver.chat_factory(chat)) for agent in AGENTS}

started = time.perf_counter()
runs = simulate(conn, cohort, 0, 0, graphs, sink, observers=[trulens, langfuse], scenarios=True)
print(f"{cohort}: {len(runs)} runs in {time.perf_counter() - started:.0f}s; streamed {sink.offsets}")

In [ ]:
# Langfuse: read every trace back, wait for its own judges, load traces and scores into
# Snowflake, and post business_correct beside them for Score Analytics.
langfuse.flush()
print("Langfuse export:", langfuse.export(conn, runs))

# TruLens: wait for each live run to finish ingesting, then compute the judge metrics in
# Snowflake plus business_correct (read from RUN_EVIDENCE) in the same call.
from refund_agent.observers import TRULENS_METRICS
truth_metric = TruLensObserver.business_correct_metric(conn, cohort)
trulens.flush()
for agent in AGENTS:
    _, app = trulens.apps[agent]
    run = app.get_run(trulens.run_name(cohort, agent))
    for _ in range(30):
        status = str(run.get_status())
        if "INVOCATION_COMPLETED" in status or "PARTIALLY" in status:
            break
        time.sleep(10)
    print(f"TruLens {agent}: {status} ->", run.compute_metrics(list(TRULENS_METRICS) + [truth_metric]))

## 3. Join to orders, policies and payments

`REFUND_AGENT_LAB.DEMO.RUN_EVIDENCE` is the business oracle. It never uses an observer score. For each run it finds:

- the policy that was effective at decision time;
- the order's age, whether the payment settled and whether the card processor accepts refunds;
- what that agent had already refunded for the order.

From those it works out what a correct agent would have refunded (`EXPECTED_EFFECT_CENTS`) and compares it with what actually moved.

In [ ]:
def sql(statement, *params):
    cur = conn.cursor()
    cur.execute(statement, params)
    return pd.DataFrame(cur.fetchall(), columns=[d[0].lower() for d in cur.description])

evidence = sql("""
SELECT TRIAL, VARIANT, AGE_DAYS, PAYMENT_STATUS, PROCESSOR_STATUS, AMOUNT_CENTS, ORDER_TOTAL_CENTS,
       POLICY_ID, AGENT_POLICY_ID, TOOL_STATUS, LEDGER_EFFECT_CENTS, EXPECTED_EFFECT_CENTS, CLAIM,
       BUSINESS_CORRECT
FROM REFUND_AGENT_LAB.DEMO.RUN_EVIDENCE WHERE COHORT_ID = ? ORDER BY TRIAL, VARIANT""", cohort)
evidence.head(12)

## 4. Find the false successes

A false success is a reply that tells the customer "refunded" when the refund didn't post for the requested amount. `CUSTOMER_MISINFORMED` is broader: the reply contradicts what happened to the money. `MONEY_AT_RISK_CENTS` is money that moved but shouldn't have.

In [ ]:
sql("""
SELECT TRIAL, VARIANT, TOOL_STATUS, LEDGER_EFFECT_CENTS, CLAIM, FALSE_SUCCESS,
       CUSTOMER_MISINFORMED, MONEY_AT_RISK_CENTS, LEFT(REPLY, 90) AS REPLY
FROM REFUND_AGENT_LAB.DEMO.RUN_EVIDENCE
WHERE COHORT_ID = ? AND (FALSE_SUCCESS OR CUSTOMER_MISINFORMED OR MONEY_AT_RISK_CENTS > 0)
ORDER BY VARIANT, TRIAL""", cohort)

## 5. Diagnose the root cause

`ROOT_CAUSE` names the first explanation the recorded facts support:

- `stale_data`: the agent used an outdated policy and refunded outside the live window.
- `tool`: the money moved doesn't match the request, for example a whole-order refund on a partial request or a refund on an unsettled payment.
- `reply`: the reply claims "refunded" when nothing moved (reply template v1).
- `model_error`: the decision contradicts the facts the agent was given.

The trace explains the *how*. TruLens and Langfuse each hold the step-by-step spans and model calls for any run, joined by the same run ID.

In [ ]:
causes = sql("""
SELECT VARIANT, COALESCE(ROOT_CAUSE, 'correct') AS ROOT_CAUSE, COUNT(*) AS RUNS,
       SUM(MONEY_AT_RISK_CENTS) / 100 AS MONEY_AT_RISK_USD, COUNT_IF(CUSTOMER_MISINFORMED) AS MISINFORMED
FROM REFUND_AGENT_LAB.DEMO.RUN_EVIDENCE WHERE COHORT_ID = ? GROUP BY ALL ORDER BY VARIANT, RUNS DESC""", cohort)
display(causes)

# The same run in both observers: Langfuse observations and the TruLens record.
example = sql("""SELECT RUN_ID FROM REFUND_AGENT_LAB.DEMO.RUN_EVIDENCE
                 WHERE COHORT_ID = ? AND ROOT_CAUSE IS NOT NULL ORDER BY TRIAL LIMIT 1""", cohort)
if not example.empty:
    run_id = example.run_id[0]
    display(sql("""SELECT TYPE, NAME, MODEL, INPUT_TOKENS, OUTPUT_TOKENS FROM REFUND_AGENT_LAB.DEMO.LANGFUSE_TRACES
                   WHERE RUN_ID = ? ORDER BY START_TIME""", run_id))
    display(sql("""SELECT METRIC, JUDGE, SCORE FROM REFUND_AGENT_LAB.DEMO.TRULENS_SCORES WHERE RUN_ID = ?""", run_id))

## 6. Compare: pick a winner, then reveal

The leaderboard runs on the interactive warehouse `REFUND_LAB_IWH`, which queries the standard tables directly with a 5-second statement cap and falls back to your standard warehouse (`REFUND_LAB_WAREHOUSE`). Resuming it starts billing, and it can't auto-suspend in under 24 hours, so this cell suspends it again.

Look at the cost, latency, self-reported success and judge columns and pick an agent before reading the business-truth columns.

In [ ]:
cur = conn.cursor()
cur.execute("USE WAREHOUSE REFUND_LAB_IWH")
cur.execute((Path.cwd().parent / "sql/leaderboard.sql").read_text().rstrip().rstrip(";"), (cohort,))
board = pd.DataFrame(cur.fetchall(), columns=[d[0].lower() for d in cur.description])
leaderboard_query = cur.sfqid
cur.execute(f"USE WAREHOUSE {config.WAREHOUSE}")
cur.execute("ALTER WAREHOUSE REFUND_LAB_IWH SUSPEND")
ran_on = sql("""SELECT WAREHOUSE_NAME, WAREHOUSE_TYPE FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY_BY_SESSION())
                WHERE QUERY_ID = ?""", leaderboard_query)
print("Leaderboard ran on:", ran_on.to_dict("records"))

tl = sql("""SELECT RUN_ID, METRIC, SCORE FROM REFUND_AGENT_LAB.DEMO.TRULENS_SCORES
            WHERE STARTSWITH(TRULENS_RUN, ?)""", f"{cohort}_")
tl = tl.merge(pd.DataFrame(runs)[["run_id", "variant"]], on="run_id").pivot_table(
    index="variant", columns="metric", values="score").add_prefix("trulens_").round(2)
board.set_index("agent").join(tl)

### What the verification run showed

These are the results of the scenario-set simulation `APP_966D3009EB5C4CF4` (9 orders, 10 requests, 30 runs, Claude agents, both observers judging with `openai-gpt-5.4-mini`). Your numbers can differ slightly: the flaws are planted, but each model decision is live.

| Agent | Microcredits / request | Self-reported success | Business-correct | Money at risk | Customers misinformed |
| --- | --- | --- | --- | --- | --- |
| Lean | **98** | **60%** | 70% | **$140.50** | 0 |
| Baseline | 1,781 | 50% | 80% | $48.00 | **1** |
| Fixed | 1,503 | 40% | **100%** | $0 | 0 |

- **Cost** picks Lean, which refunded both old-window orders under its stale 30-day policy, and the whole order on a partial request.
- **Self-reported success** picks Lean, then Baseline, which told the customer with a declined card that the refund was complete (its reply template doesn't check the tool result).
- **Only business truth** picks Fixed. Fixed reports success least often because most of these requests should be declined.

## TruLens vs Langfuse

Both observed the same runs, but they differ in practice. Everything in this table was observed while building this lab.

| | TruLens (Snowflake AI Observability) | Langfuse (self-hosted on SPCS) |
| --- | --- | --- |
| **Setup** | `pip install` the TruLens packages; register an External Agent; one live run per agent | Six containers on SPCS (web, worker, Postgres, Redis, ClickHouse, MinIO); see `langfuse_spcs/README.md` |
| **Where data lives** | Snowflake event table, queried with `GET_AI_OBSERVABILITY_EVENTS` | ClickHouse and Postgres inside the Langfuse service; the lab exports to `LANGFUSE_TRACES` and `LANGFUSE_SCORES` |
| **Trace detail** | The spans you instrument: here the request, context and reply per run | Every LangChain step automatically: graph nodes, model calls, tokens (7 observations per run) |
| **Judge** | Native, server-side in Snowflake (`answer_relevance`, `groundedness`) with Cortex `openai-gpt-5.4-mini`, set when each run is created | Native evaluators (`answer_relevance`, `groundedness`) run in the Langfuse worker on the same `openai-gpt-5.4-mini`, reached through Cortex's OpenAI-compatible REST API with a scoped 7-day token and an external access integration. They score a `judge_input` step carrying the same facts TruLens sees |
| **Score reads** | SQL over the event table | `GET /api/public/v3/scores`; the export copies them to `LANGFUSE_SCORES` |
| **Joining to business data** | Same account: one SQL join on run ID | After export to Snowflake, the same join |
| **Gotchas met** | A live run ingests once; an untraced model call inside a run breaks ingestion | Ingress needs a Snowflake token; internal SPCS hostnames can't be used in egress network rules |
| **Cost and latency per call** | Cost spans per model call (tokens, AI credits) recorded by the lab's wrapper, timed by TruLens | Computed by Langfuse from registered Cortex prices; matches the lab and TruLens exactly |
| **Cost to run** | Judge tokens through Cortex, plus event-table storage | A `CPU_X64_M` compute pool while running (an idle task suspends it after 30 minutes), block storage and judge tokens |

In [ ]:
# How well does each judge agree with business truth? Each judge is a classifier: a score
# >= 0.5 is a pass, predicting "business-correct". The _LO/_HI columns are 95% intervals.
sql("""
SELECT TOOL, METRIC, N, PASS_CORRECT, PASS_WRONG, FAIL_CORRECT, FAIL_WRONG,
       PRECISION, PRECISION_LO, PRECISION_HI, RECALL, F1, KAPPA, KAPPA_LO, KAPPA_HI,
       FALSE_PASS_RATE
FROM REFUND_AGENT_LAB.DEMO.JUDGE_AGREEMENT WHERE COHORT_ID = ? ORDER BY METRIC, TOOL""", cohort)

With 30 runs the intervals are too wide to rank judges: in the verification scenario run (`APP_966D3009EB5C4CF4`), Langfuse groundedness had a kappa of 0.29 with an interval from -0.35 to 0.94. The 300-run eval set (`scripts/run_simulation.py --eval`, simulation `APP_4B0A5F583B984FA5`: 246 correct runs, 54 wrong) gives usable numbers:

| Judge | Precision | Recall | F1 | Kappa | Wrong runs passed |
| --- | --- | --- | --- | --- | --- |
| Langfuse groundedness | 0.86 [0.82, 0.90] | 1.00 | 0.92 | 0.36 [0.17, 0.54] | 40 of 54 |
| TruLens groundedness | 0.85 [0.81, 0.89] | 0.95 | 0.90 | 0.26 [0.08, 0.44] | 40 of 54 |
| TruLens answer relevance | 0.85 [0.80, 0.89] | 0.90 | 0.87 | 0.20 [0.02, 0.37] | 39 of 54 |
| Langfuse answer relevance | 0.82 [0.77, 0.86] | 0.95 | 0.88 | -0.04 [-0.26, 0.19] | 53 of 54 |

Breaking the wrong runs down by root cause shows what groundedness misses:

| Judge | False replies passed | Stale-policy refunds passed | Whole-order refunds passed |
| --- | --- | --- | --- |
| TruLens groundedness | 0 of 12 | 18 of 18 | 22 of 24 |
| Langfuse groundedness | 0 of 12 | 18 of 18 | 22 of 24 |

Groundedness catches every reply that contradicts the facts. It can't catch an action that was wrong but described accurately, because the reply matches what the agent did. Only business truth catches those.

## Cleanup

By default this only suspends what bills by the hour: the interactive warehouse (already suspended in step 6) and the Langfuse service. Set `FULL_TEARDOWN = True` to remove everything the lab created, in dependency order. That includes the database, which holds every table, secret and the app. Run it as ACCOUNTADMIN.

In [ ]:
FULL_TEARDOWN = False  # True removes every object the lab created

cleanup = [
    "ALTER WAREHOUSE REFUND_LAB_IWH SUSPEND",                      # no-op if already suspended
    "CALL REFUND_AGENT_LAB.LANGFUSE.MANAGE_LANGFUSE('suspend','http://localhost:3000')",
]
if FULL_TEARDOWN:
    cleanup += [
        # Stop Langfuse first: a compute pool can't be dropped while services run on it.
        "ALTER TASK REFUND_AGENT_LAB.LANGFUSE.LANGFUSE_IDLE_SUSPEND SUSPEND",
        "ALTER COMPUTE POOL LANGFUSE_POOL STOP ALL",
        "DROP COMPUTE POOL IF EXISTS LANGFUSE_POOL",
        "DROP WAREHOUSE IF EXISTS REFUND_LAB_IWH",
        # Account-level identities and integrations (not inside the database).
        "DROP USER IF EXISTS REFUND_LAB_JUDGE",
        "DROP USER IF EXISTS REFUND_LAB_STREAMER",
        "DROP NETWORK POLICY IF EXISTS REFUND_LAB_JUDGE_POLICY",
        "DROP INTEGRATION IF EXISTS LANGFUSE_CORTEX_EAI",
        "DROP INTEGRATION IF EXISTS REFUND_LAB_APP_EAI",
        # Tables, views, secrets, network rules, the app and the External Agent.
        "DROP DATABASE IF EXISTS REFUND_AGENT_LAB",
        "DROP ROLE IF EXISTS REFUND_LAB_JUDGE_ROLE",
        "DROP ROLE IF EXISTS REFUND_LAB_STREAM_ROLE",
        "DROP ROLE IF EXISTS LANGFUSE_ROLE",
    ]
for statement in cleanup:
    try:
        conn.cursor().execute(statement)
        print("ok:", statement)
    except Exception as error:
        print("skipped:", statement, "-", str(error).splitlines()[0])